# 02 – Production volume corrections (ecoinvent 3.8 cutoff)

This notebook applies documented corrections to production volumes in
ecoinvent 3.8 cutoff for selected processes where default values are
known to be inconsistent with recent literature.

⚠️ **Warning**  
This notebook **modifies the ecoinvent database in place**.
It should be executed **once** per project setup.

Corrections applied:
1. Nitric acid production (global production volume)
2. Heat production at hard coal industrial furnace (1–10 MW)
3. Treatment of used refrigerant R-12, venting

In [ ]:
import bw2data as bd

In [5]:
bd.projects.set_current('Normalization')

In [ ]:
ei38cut = bd.Database("ecoinvent 3.8 cutoff")

## Correcting production volumes
### Nitric acid
Production volume for nitric acid is way to high with 26882 mio tones. Recent reports report a value of around [57 mio tones](https://www.chemanalyst.com/industry-report/nitric-acid-market-615) . Hence, the values for the nitric acid producers will be decreased according to the found number. According to ecoinvent we use the same share as:
- RER, weight = 0.3055
- RNA, weight = 0.2649
- CN, weight = 0.143
- RLA, weight = 0.08678
- SAS, weight = 0.04158
- FSU, weight = 0.0377
- RAF, weight = 0.03489
- UN-SEASIA, weight = 0.03477
- RME, weight = 0.02732
- UN-OCEANIA, weight = 0.02348

In [7]:
# Regional shares from ecoinvent (kept proportional)
weight_dict = {
    "RER w/o RU": 0.3055,
    "RNA": 0.2649,
    "CN": 0.143,
    "RLA": 0.08678,
    "SAS": 0.04158,
    "RU": 0.0377,
    "RAF": 0.03489,
    "UN-SEASIA": 0.03477,
    "RoW": 0.02732,
    "UN-OCEANIA": 0.02348
}

# Global nitric acid production volume
# Literature value ≈ 57 Mt nitric acid (100%)
NA_pv = 57 * 10**6 * 10**3  # kg/year

# Convert shares to absolute production volumes
weight_dict.update((k, v * NA_pv) for k, v in weight_dict.items())

In [8]:
NA_ACT_NAME = "nitric acid production, product in 50% solution state"
REF_PROD = "nitric acid, without water, in 50% solution state"

for act in ei38cut:
    if act["name"] != NA_ACT_NAME:
        continue

    for exc in act.production():
        if act["reference product"] == REF_PROD:
            # Allocation for 1 kg nitric acid
            exc["production volume"] = (
                weight_dict[act["location"]] * (1 / 1.649)
            )
        else:
            # Allocation for steam coproduct (0.649 kg)
            exc["production volume"] = (
                weight_dict[act["location"]] * (0.649 / 1.649)
            )
        exc.save()

### Hard coal heat production correction

In [9]:
HEAT_ACT_NAME = "heat production, at hard coal industrial furnace 1-10MW"
NEW_HEAT_PV = 1.24e12  # MJ/year

for act in ei38cut:
    if act["name"] == HEAT_ACT_NAME:
        for exc in act.production():
            exc["production volume"] = NEW_HEAT_PV
            exc.save()

### R-12 venting correction

In [10]:
R12_ACT_NAME = "treatment of used refrigerant R-12, venting"
NEW_R12_PV = 18_300_000  # kg (global banks, Lickley et al. 2021)

for act in ei38cut:
    if act["name"] == R12_ACT_NAME:
        for exc in act.production():
            exc["production volume"] = NEW_R12_PV
            exc.save()

## Sanity checks (lightweight)

In [11]:
# Quick check: print updated values
for act in ei38cut:
    if act["name"] in [HEAT_ACT_NAME, R12_ACT_NAME]:
        for exc in act.production():
            print(act["name"], exc["production volume"])

heat production, at hard coal industrial furnace 1-10MW 1240000000000.0
treatment of used refrigerant R-12, venting 18300000
heat production, at hard coal industrial furnace 1-10MW 1240000000000.0
heat production, at hard coal industrial furnace 1-10MW 1240000000000.0


✔ Production volume corrections applied successfully  

➡ Continue with `03_database_wide_inventory_build.ipynb`